### Setup

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# NOTE: Control randomness for reproducibility
set_seed(RANDOM_SEED)
# Initialize data processors
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()



/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5,
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[bhiravi_vaidhy, dilip_satgare, haresh_mehta, ...",USA,siddharth_randeria,Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,mel_gibson,Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[dylan_walsh, laura_linney, ernie_hudson_jr, t...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358027 (74.84%
valid: 46916 (9.81%)
test: 73461 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555944
1    0.444056
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585277
1    0.414723
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [ ]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df, is_offset=False)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358027
  Num of positive interactions: 158984 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 158984], edge_label=[158984])
Edge Index: tensor([[   0,    0,    0,  ..., 2063, 2063, 2063],
        [1102, 1186,  670,  ..., 2835,  742, 2969]])


#### Prepare prediction pool for inference/testing

In [8]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2064
Item Pool: 6959, negative sampled to 500 items for each user
Num of interactions: 2064(users) * 500(items) = 1032000


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
1031995,2063,447,0,"[15727, 2894, 10573, 253, 3728]",62,2985,"[8, 0, 0, 0, 0, 0, 0, 0]"
1031996,2063,3601,0,"[11716, 4113, 7140, 7336, 6954]",63,2854,"[1, 18, 0, 0, 0, 0, 0, 0]"
1031997,2063,7982,0,"[6627, 283, 3901, 11769, 14462]",63,1419,"[11, 0, 0, 0, 0, 0, 0, 0]"
1031998,2063,5969,0,"[11666, 10099, 4687, 6007, 6501]",63,3441,"[8, 15, 0, 0, 0, 0, 0, 0]"
1031999,2063,5025,0,"[3962, 7760, 5111, 6372, 4643]",62,558,"[6, 11, 14, 17, 0, 0, 0, 0]"


### Prepare DataLoader

In [9]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import UserItemPairDataset

BATCH_SIZE = 1024

train_dataset = UserItemPairDataset(encoded_train_df)
valid_dataset = UserItemPairDataset(encoded_valid_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 358027
valid data count: 46916
test data count: 1032000


### Configure Model (LightningModule)

In [10]:
from lightning_models.bce.gcn_cf_bce_rec import GCNRecCF

EMB_DIM = 32
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 3

model = GCNRecCF(
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,  # shape [2, num_edges]
    dim_id=EMB_DIM,
    num_layers=NUM_LAYERS,
    concat=True,
    lr=LR,
)


### Configure Trainer and Experiment

In [11]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "gcn-bce-exp"
RUN_NAME = "gcn-baseline-test7-9"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME)
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_f1",
    monitor_mode="max",
    hyper_param_str=f"emb_dim={EMB_DIM}-num_layers={NUM_LAYERS}-lr={LR}-batch_size={BATCH_SIZE}-epochs={EPOCHS}",
)

In [12]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='cpu',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


### Train Model

In [13]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/emma/10TB/home/bilab_archive/Bai/DPRecSys/experiments/test_checkpoints/gcn-bce-exp exists and is not empty.

  | Name      | Type            | Params | Mode 
------------------------------------------------------
0 | gcn_model | GraphConvModule | 376 K  | train
------------------------------------------------------
376 K     Trainable params
0         Non-trainable params
376 K     Total params
1.504     Total estimated model params size (MB)
43        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 4, global step 1750: 'val_f1' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_f1 did not improve in the last 5 records. Best score: 0.561. Signaling Trainer to stop.
Epoch 5, global step 2100: 'val_f1' was not in top 1


🏃 View run gcn-baseline-test7-9 at: http://140.112.106.216:3683/#/experiments/3/runs/c59972f423354e0aa1a5743fa250e12f
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/3


### Inference

In [14]:
# NOTE: the inference model MUST be the same as the training model
best_model_experiment_name = "gcn-bce-exp"
best_model_checkpoint_path = "gcn-baseline-test7-9-emb_dim=32-num_layers=3-lr=0.001-batch_size=1024-epochs=50-best-checkpoint-epoch=00-val_f1=0.56.ckpt"
best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"

model = GCNRecCF.load_from_checkpoint(
    checkpoint_path=best_model_path,
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,
    dim_id=EMB_DIM,
    num_layers=NUM_LAYERS,
    concat=True,
    lr=LR,
)


In [15]:
# start inference
trainer.test(model=model, dataloaders=test_loader)

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │   0.032355621457099915    │
│          test_f1          │   0.057495467364788055    │
│         test_loss         │        283205.3125        │
│         test_prec         │    0.02959882840514183    │
│         test_rec          │    0.9997702240943909     │
└───────────────────────────┴───────────────────────────┘

🏃 View run gcn-baseline-test7-9 at: http://140.112.106.216:3683/#/experiments/3/runs/c59972f423354e0aa1a5743fa250e12f
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/3


[{'test_loss': 283205.3125,
  'test_acc': 0.032355621457099915,
  'test_prec': 0.02959882840514183,
  'test_rec': 0.9997702240943909,
  'test_f1': 0.057495467364788055}]

- gcn-bce-exp-gcn-baseline-test2-best-checkpoint-epoch=04-val_f1=0.45.ckpt
<details>
K=500

- [{'test_loss': 364.225341796875,
- 'test_acc': 0.7884315252304077,
- 'test_prec': 0.05633168667554855,
- 'test_rec': 0.39781633019447327,
- 'test_f1': 0.09868881106376648}]

K=250

- [{'test_loss': 372.6000061035156,
- 'test_acc': 0.7848190665245056,
- 'test_prec': 0.07433757930994034,
- 'test_rec': 0.3977948725223541,
- 'test_f1': 0.1252661496400833}]
</details>

- gcn-bce-exp-gcn-baseline-test3-best-checkpoint-epoch=00-val_f1=0.57.ckpt
<details>
K=500

- [{'test_loss': 12384.9208984375,
- 'test_acc': 0.5445107221603394,
- 'test_prec': 0.049183208495378494,
- 'test_rec': 0.7988131046295166,
- 'test_f1': 0.09266123175621033}]
</details>

In [16]:
model.test_results

{'user': tensor([   0,    0,    0,  ..., 2063, 2063, 2063]),
 'item': tensor([7977, 4839,  979,  ..., 7982, 5969, 5025]),
 'score': tensor([596816.5000,  29566.3516, 428610.1250,  ...,   8061.4238,
          33179.3047,  39050.2422]),
 'label': tensor([1., 0., 1.,  ..., 0., 0., 0.]),
 'metric': {'test_loss': 283205.3125,
  'test_acc': 0.03235562015503876,
  'test_prec': 0.029598828446515804,
  'test_rec': 0.999770235672553,
  'test_f1': 0.057495467317019766},
 'user_emb': tensor([[-1.1270e+02, -6.9671e+00, -5.5779e+01,  ...,  2.5014e+03,
           8.6392e+03,  1.0669e+04],
         [-5.3454e+01, -3.3041e+00, -2.6549e+01,  ...,  1.1878e+03,
           4.0967e+03,  5.0585e+03],
         [-9.6463e+00, -5.9032e-01, -4.7763e+00,  ...,  2.1320e+02,
           7.3968e+02,  9.1195e+02],
         ...,
         [-3.1277e+02, -1.9300e+01, -1.5477e+02,  ...,  6.9187e+03,
           2.3949e+04,  2.9583e+04],
         [-5.6408e+02, -3.4976e+01, -2.8080e+02,  ...,  1.2530e+04,
           4.3119e+04,

In [17]:
eval_df = evaluator.prepare_evaluation_data(model.test_results)
eval_df

,user,rec_items,gt_items
0,0,"[2090, 281, 4142, 2419, 5946, 4932, 316, 3550,...","[7977, 979, 97, 2419, 2090]"
1,1,"[3550, 968, 1364, 7999, 2501, 972, 871, 2444, ...","[3392, 5801, 6570, 8192]"
2,2,"[2419, 1605, 1344, 4045, 945, 991, 1354, 8103,...","[7978, 5768]"
3,3,"[260, 231, 49, 907, 1035, 1009, 7908, 969, 330...","[969, 3246, 3303, 2066, 6880, 7908, 7979, 2932..."
4,4,"[1242, 5941, 1354, 7599, 8081, 4970, 7805, 339...","[1492, 1139, 4085, 8160, 6253, 1380, 4395, 8212]"
...,...,...,...
2059,2059,"[1035, 870, 961, 4493, 1326, 4964, 5941, 1354,...","[1267, 2606, 1584, 4493, 1488, 4964, 3500, 836..."
2060,2060,"[688, 1014, 1865, 1344, 5566, 1539, 621, 1275,...","[1596, 1790, 5627, 977, 6914, 8311, 1587, 823,..."
2061,2061,"[688, 49, 5706, 7546, 6170, 1009, 942, 480, 40...","[8140, 8017, 8126, 5941, 8214, 8075]"
2062,2062,"[4932, 3289, 0, 1344, 258, 1326, 1395, 7999, 6...","[3289, 4932, 645, 1496, 1889, 1326]"


In [18]:
eval_score_df = evaluator.evaluate(eval_df, K=5)
eval_score_df = evaluator.evaluate(eval_score_df, K=10)
eval_score_df = evaluator.evaluate(eval_score_df, K=20)
eval_score_df.describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.339828,0.092032,0.175775,0.398099,0.169261,0.173692,0.437006,0.281375,0.156323
std,595.969798,0.372615,0.157939,0.216871,0.326700,0.213544,0.178432,0.281109,0.264535,0.141764
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.262650,0.072735,0.050000
50%,1031.500000,0.386853,0.015625,0.200000,0.421773,0.100000,0.100000,0.445859,0.214286,0.100000
75%,1547.250000,0.630930,0.125000,0.400000,0.630930,0.250000,0.300000,0.636540,0.416667,0.250000
max,2063.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.900000,1.000000,1.000000,0.800000
